# Builder Design Pattern

Here is the Builder Design Pattern implemented in a modern, Pythonic way (using Method Chaining and Dataclasses).

### The Concept
The Builder Pattern is used to construct complex objects step-by-step. Unlike the Factory (which creates an object in one shot), the Builder allows you to configure an object with various options before finally "building" it.

Think of it like ordering a custom sandwich (sub):
- Choose Bread
- Choose Cheese
- Add Veggies
- Add Sauce
- Wrap it up (Build)

## THE PRODUCT

In [10]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class Computer:
    """The complex object we are building."""
    cpu: str = "Standard CPU"
    ram: str = "8GB"
    storage: str = "256GB SSD"
    gpu: Optional[str] = None
    cooling: str = "Air Cooling"

    def __str__(self) -> str:
        return (f"🖥️  Computer Specs:\n"
                f"   ├── CPU:     {self.cpu}\n"
                f"   ├── RAM:     {self.ram}\n"
                f"   ├── Storage: {self.storage}\n"
                f"   ├── GPU:     {self.gpu or 'Integrated Graphics'}\n"
                f"   └── Cooling: {self.cooling}")

## THE BUILDER

In [11]:
from typing import Self

class ComputerBuilder:
    """
    The Builder allows step-by-step construction.
    Methods return 'Self' to allow Method Chaining (fluent interface).
    """
    def __init__(self):
        # We start with a blank/default computer
        self._computer = Computer()

    def set_cpu(self, model: str) -> Self:
        self._computer.cpu = model
        return self

    def set_ram(self, size: str) -> Self:
        self._computer.ram = size
        return self

    def set_storage(self, drive: str) -> Self:
        self._computer.storage = drive
        return self

    def set_gpu(self, model: str) -> Self:
        self._computer.gpu = model
        return self

    def set_water_cooling(self) -> Self:
        self._computer.cooling = "Liquid Cooling Loop"
        return self

    def build(self) -> Computer:
        """Finalizes construction and returns the object."""
        # You could add validation logic here (e.g., check compatibility)
        built_pc = self._computer
        # Reset the builder for the next build (optional)
        self._computer = Computer() 
        return built_pc

ERROR! Session/line number was not unique in database. History logging moved to new session 5


## THE DIRECTOR (Optional)

In [12]:
class PCShop:
    """
    The Director uses the builder to make standard configurations.
    Useful if you have pre-defined 'recipes'.
    """
    def make_gaming_pc(self, builder: ComputerBuilder) -> Computer:
        return (builder
                .set_cpu("Intel Core i9")
                .set_ram("32GB DDR5")
                .set_gpu("NVIDIA RTX 4090")
                .set_water_cooling()
                .set_storage("2TB NVMe SSD")
                .build())

    def make_office_pc(self, builder: ComputerBuilder) -> Computer:
        return (builder
                .set_cpu("Intel Core i5")
                .set_ram("16GB")
                .build())

## CLIENT CODE

In [13]:
def main():
    builder = ComputerBuilder()
    
    # SCENARIO A: Custom Build (Method Chaining)
    print("--- 1. Custom Build (The Pythonic Way) ---")
    my_custom_pc = (builder
                    .set_cpu("AMD Ryzen 7")
                    .set_gpu("AMD Radeon RX 7900")
                    .set_ram("64GB")
                    .build())
    print(my_custom_pc)
    print()

    # SCENARIO B: Director Build (Pre-set Recipes)
    print("--- 2. Director Build (The Recipe Way) ---")
    shop = PCShop()
    
    gaming_rig = shop.make_gaming_pc(builder)
    print(gaming_rig)

if __name__ == "__main__":
    main()

--- 1. Custom Build (The Pythonic Way) ---
🖥️  Computer Specs:
   ├── CPU:     AMD Ryzen 7
   ├── RAM:     64GB
   ├── Storage: 256GB SSD
   ├── GPU:     AMD Radeon RX 7900
   └── Cooling: Air Cooling

--- 2. Director Build (The Recipe Way) ---
🖥️  Computer Specs:
   ├── CPU:     Intel Core i9
   ├── RAM:     32GB DDR5
   ├── Storage: 2TB NVMe SSD
   ├── GPU:     NVIDIA RTX 4090
   └── Cooling: Liquid Cooling Loop


#### Why is this Pythonic?

- `typing.Self`: Introduced in Python 3.11, this allows the type checker to know that `set_cpu()` returns the Builder instance itself, enabling the nice dot-notation chain `(.set_cpu().set_ram())`.
- Dataclasses: The `Product` is a simple dataclass, removing the need for a huge `__init__` method full of boilerplate assignments.
- Method Chaining: This mimics popular Python libraries (like Pandas or SQLAlchemy) where you chain operations together.

#### When to use this?
- Use it when: Your object has 5+ parameters, many of which are optional, or if the construction process involves complex logic (like "if you add a GPU, you must upgrade the Power Supply").
- Don't use it when: Your object only needs 2-3 arguments. Just use `Computer(cpu="i7", ram="16GB") `(standard constructor).
- Would you like to see the Singleton Pattern next (ensuring only one instance of a class exists)?

## Here is the "True Pythonic" way to solve the Builder problem.

#### The Truth About Python Builders
In languages like Java or C++, the Builder pattern is necessary because constructors cannot easily handle optional parameters (you end up with "Telescoping Constructors").

In Python, we have Named Arguments and Default Values. </br>
Therefore, the most Pythonic Builder is usually just the Class itself (specifically a dataclass). It is cleaner, faster, and 100% built-in.

### THE BUILDER IS THE PRODUCT

In [14]:
from dataclasses import dataclass, field
from typing import Optional

@dataclass(frozen=True)
class Computer:
    """
    In Python, the 'Builder' pattern is built into the language features:
    1. Keyword Arguments (naming parameters)
    2. Default Values (optional parameters)
    """
    # Required parts (Must be provided)
    cpu: str
    
    # Optional parts (Have defaults)
    ram: str = "8GB"
    storage: str = "256GB SSD"
    gpu: Optional[str] = None
    cooling: str = "Air Cooling"

    # Derived/Computed fields (Logic that usually lives in a Builder 'build()' method)
    # can live in __post_init__. Since we used frozen=True, we have a workaround,
    # or simpler: just use a property.
    @property
    def specs(self) -> str:
        return (f"🖥️  Spec Sheet:\n"
                f"   CPU: {self.cpu} | RAM: {self.ram} | GPU: {self.gpu or 'Integrated'}\n"
                f"   Cooling: {self.cooling}")

    # ==========================================
    # 2. "DIRECTOR" LOGIC (Factory Methods)
    # ==========================================
    # Instead of a separate 'Director' class, use Class Methods.
    
    @classmethod
    def create_gaming_rig(cls) -> "Computer":
        """Pre-configured Gaming Setup"""
        return cls(
            cpu="Intel Core i9",
            ram="32GB DDR5",
            gpu="NVIDIA RTX 4090",
            cooling="Liquid Loop",
            storage="2TB NVMe"
        )

    @classmethod
    def create_office_pc(cls) -> "Computer":
        """Pre-configured Office Setup"""
        return cls(
            cpu="Intel Core i5",
            ram="16GB"
            # Uses default storage, gpu, cooling
        )

### CLIENT CODE

In [15]:
def main():
    print("--- 1. Custom Build (Named Arguments) ---")
    # This replaces: builder.set_cpu(...).set_ram(...).build()
    custom_pc = Computer(
        cpu="AMD Ryzen 7",
        ram="64GB",
        gpu="Radeon RX 7900"
    )
    print(custom_pc.specs)
    print()

    print("--- 2. Preset Build (Class Method) ---")
    gaming_pc = Computer.create_gaming_rig()
    print(gaming_pc.specs)

if __name__ == "__main__":
    main()

--- 1. Custom Build (Named Arguments) ---
🖥️  Spec Sheet:
   CPU: AMD Ryzen 7 | RAM: 64GB | GPU: Radeon RX 7900
   Cooling: Air Cooling

--- 2. Preset Build (Class Method) ---
🖥️  Spec Sheet:
   CPU: Intel Core i9 | RAM: 32GB DDR5 | GPU: NVIDIA RTX 4090
   Cooling: Liquid Loop
